# Phase 1 — Multi-Source Data Pipeline
**Project:** R26-DS-012 | **Component:** C3 | **Student:** Seneviratne K.A.U.A. | IT22093950

## Data sources
| Dataset | Records | Labels | Key features |
|---------|---------|--------|--------------|
| NHANES 2017–2020 (USA) | ~2,705 | GAD-7 severity labels | Demographics |
| Colombia Mendeley 2022 | ~2,657 | **Real GAD-7 scores** | Real GAD-7 + PSS-10 + PHQ-9 |

## Pipeline
| Cell | What it does | Output |
|------|-------------|--------|
| 1 | Install & import | — |
| 2 | Load NHANES dataset (nhanes_c3_dataset.csv) | — |
| 3 | Engineer 13 features from NHANES | — |
| 4 | Load Colombia Raw_data.xlsx → map 13 features | — |
| 5 | Combine NHANES + Colombia | combined_c3_raw.csv |
| 6 | Stratified split (by source + class) | — |
| 7 | SMOTE balancing on training fold only | — |
| 8 | Figures | figure1_pipeline.png |
| 9 | Save all outputs | nhanes_c3_raw.csv, colombia_c3_real.csv, combined_c3_raw.csv, combined_c3_balanced.csv |
| 10 | Download | — |

In [ ]:
# ================================================================
# CELL 1 — Install & Import
# ================================================================
!pip install imbalanced-learn scikit-learn matplotlib seaborn pandas numpy openpyxl --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, json, os
from collections import Counter
from pathlib import Path
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import openpyxl

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'Low': '#22C55E', 'Medium': '#F59E0B', 'High': '#EF4444'}
RISK_LABELS = {0: 'Low', 1: 'Medium', 2: 'High'}

# The 13 feature columns — order is fixed and must match Phase 2
FEATURE_COLS = [
    'age_norm',               # F1
    'gender_enc',             # F2
    'marital_enc',            # F3
    'education_enc',          # F4
    'income_enc',             # F5
    'physiological_risk',     # F6  — C1 wearable proxy
    'behavioral_risk',        # F7  — C2 behavioral proxy
    'textual_risk',           # F8  — C4 NLP proxy
    'composite_risk',         # F9  — derived
    'risk_tier_enc',          # F10 — previous session
    'interaction_count_norm', # F11
    'last_reward_norm',       # F12
    'escalation_count_norm',  # F13
]
TARGET_COL = 'risk_tier'

print('Setup complete ✓')
print(f'Feature schema: {len(FEATURE_COLS)} features frozen')

In [ ]:
# ================================================================
# CELL 2 — Load NHANES Dataset
# ================================================================
from google.colab import files

print('Upload nhanes_c3_dataset.csv...')
up1 = files.upload()
nhanes_key = next(k for k in up1 if 'nhanes' in k.lower())
df_nhanes_raw = pd.read_csv(nhanes_key)

print(f'Loaded: {len(df_nhanes_raw):,} records')
print(f'Columns: {list(df_nhanes_raw.columns)}')
print()

# Verify expected columns present
required = ['age','gender_enc','marital_enc','education_enc','income_pir','gad7_total','risk_tier']
missing  = [c for c in required if c not in df_nhanes_raw.columns]
if missing:
    raise ValueError(f'Missing columns in nhanes_c3_dataset.csv: {missing}')
print('All required columns present ✓')

# Class distribution
print('\nNHANES class distribution:')
n_nh = len(df_nhanes_raw)
for t, label in [(0,'Low  (0-4)'), (1,'Medium (5-9)'), (2,'High (10+)')]:
    cnt = (df_nhanes_raw['risk_tier']==t).sum()
    print(f'  {label}: {cnt:4d} ({cnt/n_nh*100:.1f}%)')


In [ ]:
# ================================================================
# CELL 3 — Engineer 13 Features from NHANES
# ================================================================
# F6/F7/F8: risk score inputs from C1/C2/C4 subsystems.
# Derived from GAD-7 severity with individual-level noise
# to reflect real-world measurement variability.

rng = np.random.default_rng(42)
n_nh = len(df_nhanes_raw)
gad_norm = df_nhanes_raw['gad7_total'].values / 21.0

X_nhanes = pd.DataFrame({
    # Demographics (direct from NHANES)
    'age_norm':               (df_nhanes_raw['age'] - 18) / 17,
    'gender_enc':             df_nhanes_raw['gender_enc'].astype(int),
    'marital_enc':            df_nhanes_raw['marital_enc'].clip(1,3).astype(int),
    'education_enc':          df_nhanes_raw['education_enc'].clip(1,5).astype(int),
    'income_enc':             (df_nhanes_raw['income_pir'] / 5.0).clip(0, 1),

    # Risk score proxies (GAD-7 derived with individual noise)
    # Published: GAD-7 correlates strongly with HRV anomalies (F6),
    # social withdrawal / screen time (F7), and language distress markers (F8)
    'physiological_risk':     np.clip(gad_norm*0.90 + rng.normal(0, 0.05, n_nh), 0, 1),
    'behavioral_risk':        np.clip(gad_norm*0.80 + rng.normal(0, 0.05, n_nh), 0, 1),
    'textual_risk':           np.clip(gad_norm*0.75 + rng.normal(0, 0.05, n_nh), 0, 1),

    # Session history (zeros — no prior sessions in pre-training)
    'risk_tier_enc':          df_nhanes_raw['risk_tier'].astype(int),
    'interaction_count_norm': 0.0,
    'last_reward_norm':       0.0,
    'escalation_count_norm':  0.0,
})

# Add composite risk (must be computed after F6/F7/F8)
X_nhanes.insert(
    FEATURE_COLS.index('composite_risk'),
    'composite_risk',
    (
        0.25 * X_nhanes['physiological_risk'] +
        0.20 * X_nhanes['behavioral_risk']    +
        0.40 * X_nhanes['textual_risk']
    ) / 0.85
)

# Reorder to match FEATURE_COLS exactly
X_nhanes = X_nhanes[FEATURE_COLS]
y_nhanes = df_nhanes_raw['risk_tier'].astype(int).values

print(f'NHANES feature matrix: {X_nhanes.shape}')
print('Class distribution:')
for t in [0,1,2]:
    cnt = (y_nhanes==t).sum()
    print(f'  {RISK_LABELS[t]:8s}: {cnt:4d} ({cnt/len(y_nhanes)*100:.1f}%)')

# Sanity check: F9 = formula of F6/F7/F8
expected_f9 = (0.25*X_nhanes['physiological_risk']
               + 0.20*X_nhanes['behavioral_risk']
               + 0.40*X_nhanes['textual_risk']) / 0.85
max_dev = (X_nhanes['composite_risk'] - expected_f9).abs().max()
print(f'F9 consistency check: max deviation = {max_dev:.8f} ✓' if max_dev < 1e-6 else
      f'⚠ F9 inconsistency: {max_dev:.6f}')

In [ ]:
# ================================================================
# CELL 4 — Load & Map Colombia Dataset
# ================================================================
# Source: Restrepo-Henao et al. 2022
# Mendeley Data DOI: bytb22nf7m
# Real GAD-7, PSS-10, PHQ-9 scores from 3,052 undergrad students, Colombia
#
# Feature mapping:
#   F2  gender_enc         = col[1]  (1=M, 2=F)
#   F5  income_enc         = SES (1-6) / 6.0
#   F6  physiological_risk = simulated from GAD-7 tier (no wearable data available)
#   F7  behavioral_risk    = PSS-10 / 40   ← REAL validated stress score
#   F8  textual_risk       = PHQ-9  / 27   ← REAL validated depression score
#   TARGET                 = GAD-7 total → tier using Löwe 2008 cutoffs  ← REAL

print('Upload Raw_data.xlsx (Colombia Mendeley dataset)...')
up3 = files.upload()
xlsx_key = next(k for k in up3 if k.endswith('.xlsx') or k.endswith('.xls'))

wb  = openpyxl.load_workbook(xlsx_key, read_only=True)
ws  = wb.active
raw_rows = list(ws.iter_rows(min_row=3, values_only=True))  # row 1=section, 2=headers, 3+=data
print(f'Loaded {len(raw_rows)} raw rows')

# Column indices (confirmed by inspection)
COL_YEAR = 0; COL_GENDER = 1; COL_SES = 2
PSS10_IDX = list(range(16, 26))   # Q16.1–16.10 (0–4 each, max 40)
GAD7_IDX  = list(range(26, 33))   # Q17.1–17.7  (0–3 each, max 21)
PHQ9_IDX  = list(range(33, 42))   # Q18.1–18.9  (0–3 each, max 27)

rng_col = np.random.default_rng(42)
colombia_records = []
n_skipped = 0

for r in raw_rows:
    # Require year, gender, SES, complete GAD-7
    if not (r[COL_YEAR] and isinstance(r[COL_YEAR],(int,float))
            and r[COL_GENDER] and r[COL_SES]):
        n_skipped += 1; continue

    gad7_items = [r[i] for i in GAD7_IDX]
    if not all(v is not None and isinstance(v,(int,float)) for v in gad7_items):
        n_skipped += 1; continue

    age = 2022 - int(r[COL_YEAR])
    if not (18 <= age <= 35):
        n_skipped += 1; continue

    gad7_total = int(sum(gad7_items))

    # Real PSS-10 → F7 behavioral_risk
    pss_items = [r[i] for i in PSS10_IDX]
    if all(v is not None and isinstance(v,(int,float)) for v in pss_items):
        behavioral_risk = float(sum(pss_items)) / 40.0
    else:
        behavioral_risk = float(gad7_total) / 21.0 * 0.85   # fallback

    # Real PHQ-9 → F8 textual_risk
    phq_items = [r[i] for i in PHQ9_IDX]
    if all(v is not None and isinstance(v,(int,float)) for v in phq_items):
        textual_risk = float(sum(phq_items)) / 27.0
    else:
        textual_risk = float(gad7_total) / 21.0 * 0.75       # fallback

    # F6 physiological_risk — simulated from GAD-7 severity (no wearable)
    gad_norm_v = float(gad7_total) / 21.0
    physiological_risk = float(np.clip(gad_norm_v * 0.90 + rng_col.normal(0, 0.04), 0, 1))

    # Composite risk
    composite_risk = float(np.clip(
        (0.25*physiological_risk + 0.20*behavioral_risk + 0.40*textual_risk) / 0.85, 0, 1
    ))

    # Risk tier from real GAD-7 (Löwe 2008 cutoffs: 0-4=Low, 5-9=Medium, 10+=High)
    risk_tier_v = 0 if gad7_total <= 4 else (1 if gad7_total <= 9 else 2)

    colombia_records.append([
        float(age - 18) / 17,            # age_norm
        int(r[COL_GENDER]),               # gender_enc (1=M, 2=F)
        3,                                # marital_enc (students → never married)
        4,                                # education_enc (undergrad)
        float(r[COL_SES]) / 6.0,          # income_enc
        round(physiological_risk, 5),     # F6
        round(float(np.clip(behavioral_risk, 0, 1)), 5),   # F7 (real PSS-10)
        round(float(np.clip(textual_risk,   0, 1)), 5),    # F8 (real PHQ-9)
        round(composite_risk, 5),         # F9
        0,                                # risk_tier_enc (first session)
        0.0,                              # interaction_count_norm
        0.5,                              # last_reward_norm (neutral prior)
        0.0,                              # escalation_count_norm
        risk_tier_v                       # TARGET (real GAD-7)
    ])

df_colombia_full = pd.DataFrame(colombia_records,
                                 columns=FEATURE_COLS + [TARGET_COL])

X_colombia = df_colombia_full[FEATURE_COLS]
y_colombia = df_colombia_full[TARGET_COL].astype(int).values

print(f'Skipped rows (missing data/age): {n_skipped}')
print(f'Colombia records (18-35, complete): {len(df_colombia_full):,}')
print()
print('Colombia class distribution (REAL GAD-7 labels):')
for t in [0,1,2]:
    cnt = (y_colombia==t).sum()
    print(f'  {RISK_LABELS[t]:8s}: {cnt:4d} ({cnt/len(y_colombia)*100:.1f}%)')
print()
print('Real feature statistics:')
print(f'  F7 behavioral_risk (PSS-10/40): mean={X_colombia["behavioral_risk"].mean():.3f} ← REAL')
print(f'  F8 textual_risk    (PHQ-9/27) : mean={X_colombia["textual_risk"].mean():.3f} ← REAL')
print(f'  TARGET risk_tier (real GAD-7) : confirmed ✓')

In [ ]:
# ================================================================
# CELL 5 — Combine NHANES + Colombia
# ================================================================
# Both datasets use the exact same 13-feature schema.
# A 'source' column tags origin for stratification and analysis.
# F9 consistency is re-verified after combination.

# Add source tags
X_nhanes_tagged   = X_nhanes.copy();   X_nhanes_tagged['source']   = 'nhanes'
X_colombia_tagged = X_colombia.copy(); X_colombia_tagged['source'] = 'colombia'
y_all = np.concatenate([y_nhanes, y_colombia])

X_combined_tagged = pd.concat([X_nhanes_tagged, X_colombia_tagged], ignore_index=True)
X_combined = X_combined_tagged[FEATURE_COLS]   # model features only
source_all = X_combined_tagged['source'].values

print('Combined dataset summary:')
print(f'  NHANES rows    : {len(y_nhanes):,}')
print(f'  Colombia rows  : {len(y_colombia):,}')
print(f'  Combined total : {len(y_all):,}')
print()
print('Combined class distribution:')
for t in [0,1,2]:
    cnt = (y_all==t).sum()
    print(f'  {RISK_LABELS[t]:8s}: {cnt:5d} ({cnt/len(y_all)*100:.1f}%)')

# F9 consistency check on combined
expected_f9 = (0.25*X_combined['physiological_risk']
               + 0.20*X_combined['behavioral_risk']
               + 0.40*X_combined['textual_risk']) / 0.85
max_dev = (X_combined['composite_risk'] - expected_f9).abs().max()
print(f'\nF9 consistency check (combined): max deviation = {max_dev:.8f}',
      '✓' if max_dev < 1e-4 else '⚠ CHECK NEEDED')

# Null check
null_counts = X_combined.isnull().sum()
total_nulls = null_counts.sum()
print(f'Null values in combined: {total_nulls}', '✓' if total_nulls == 0 else '⚠ FIX NEEDED')

In [ ]:
# ================================================================
# CELL 6 — Stratified Train / Test Split
# ================================================================
# Split BEFORE any resampling (prevents data leakage).
#
# Stratification key = source × class
# This guarantees both NHANES and Colombia records appear in
# BOTH train and test sets — avoiding source-only confounds.
#
# Split: 80% train, 20% test (slightly larger test than Phase 2
# to accommodate the combined dataset size)

# Create a combined stratification key
strat_key = np.array([f'{s}_{t}' for s, t in zip(source_all, y_all)])

X_arr = X_combined.values.astype(np.float32)

X_train, X_test, y_train, y_test, src_train, src_test = train_test_split(
    X_arr, y_all, source_all,
    test_size=0.20,
    stratify=strat_key,
    random_state=42
)

def print_split(name, X, y, src):
    n = len(y)
    nh = (src=='nhanes').sum()
    co = (src=='colombia').sum()
    low  = (y==0).sum(); med = (y==1).sum(); high = (y==2).sum()
    print(f'  {name:8s}: {n:5d} rows | NHANES={nh} Colombia={co} | '
          f'Low={low}({low/n*100:.0f}%) Med={med}({med/n*100:.0f}%) High={high}({high/n*100:.0f}%)')

print('Stratified split (source × class):')
print_split('Train', X_train, y_train, src_train)
print_split('Test',  X_test,  y_test,  src_test)
print()

# Verify both sources present in both splits
for split_name, src in [('Train', src_train), ('Test', src_test)]:
    sources_present = set(src)
    ok = 'nhanes' in sources_present and 'colombia' in sources_present
    print(f'  {split_name} sources: {sources_present} ✓' if ok else
          f'  ⚠ {split_name} missing a source — re-check stratification')

In [ ]:
# ================================================================
# CELL 7 — SMOTE Balancing (Training Set Only)
# ================================================================
# SMOTE applied ONLY to training set.
# Test set keeps the natural (imbalanced) distribution — this gives
# realistic evaluation metrics, not inflated ones.
#
# Note: SMOTE generates interpolated records between existing data points.

print('Before SMOTE (train):', dict(sorted(Counter(y_train).items())))

smote = SMOTE(k_neighbors=5, random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('After  SMOTE (train):', dict(sorted(Counter(y_train_bal).items())))
print(f'\nTraining set: {len(y_train):,} → {len(y_train_bal):,} rows after SMOTE')
print(f'Test set    : {len(y_test):,} rows (unchanged — natural distribution)')

In [ ]:
# ================================================================
# CELL 8 — Dataset Pipeline Figure
# ================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(
    'C3 Multi-Source Data Pipeline\n'
    'NHANES 2017–2020 (regression-labelled) + Colombia 2022 (real GAD-7)',
    fontsize=13, fontweight='bold'
)

t_names = ['Low\n(0–4)', 'Medium\n(5–9)', 'High\n(10+)']
clr     = [COLORS['Low'], COLORS['Medium'], COLORS['High']]

# 1. NHANES distribution
nh_c = [int((y_nhanes==t).sum()) for t in range(3)]
axes[0,0].bar(t_names, nh_c, color=clr, edgecolor='white', width=0.5, alpha=0.9)
axes[0,0].set_title('NHANES\n(regression-based GAD-7 labels)', fontweight='bold')
axes[0,0].set_ylabel('Count')
for j, v in enumerate(nh_c):
    axes[0,0].text(j, v+5, f'{v}\n{v/len(y_nhanes)*100:.0f}%', ha='center', fontsize=9)

# 2. Colombia distribution
co_c = [int((y_colombia==t).sum()) for t in range(3)]
axes[0,1].bar(t_names, co_c, color=clr, edgecolor='white', width=0.5, alpha=0.9)
axes[0,1].set_title('Colombia Mendeley 2022\n(real GAD-7 labels ✓)', fontweight='bold')
axes[0,1].set_ylabel('Count')
for j, v in enumerate(co_c):
    axes[0,1].text(j, v+5, f'{v}\n{v/len(y_colombia)*100:.0f}%', ha='center', fontsize=9)

# 3. Combined training (before SMOTE)
comb_c = [int((y_train==t).sum()) for t in range(3)]
axes[0,2].bar(t_names, comb_c, color=clr, edgecolor='white', width=0.5, alpha=0.9)
axes[0,2].set_title('Combined Training Set\n(before SMOTE)', fontweight='bold')
axes[0,2].set_ylabel('Count')
for j, v in enumerate(comb_c):
    axes[0,2].text(j, v+5, f'{v}\n{v/len(y_train)*100:.0f}%', ha='center', fontsize=9)

# 4. After SMOTE
bal_c = [int((y_train_bal==t).sum()) for t in range(3)]
axes[1,0].bar(t_names, bal_c, color=clr, edgecolor='white', width=0.5, alpha=0.9)
axes[1,0].set_title('Training Set After SMOTE\n(balanced)', fontweight='bold')
axes[1,0].set_ylabel('Count')
for j, v in enumerate(bal_c):
    axes[1,0].text(j, v+10, f'{v}', ha='center', fontsize=10, fontweight='bold')

# 5. Test set
test_c = [int((y_test==t).sum()) for t in range(3)]
axes[1,1].bar(t_names, test_c, color=clr, edgecolor='white', width=0.5, alpha=0.9)
axes[1,1].set_title('Test Set\n(natural distribution — not resampled)', fontweight='bold')
axes[1,1].set_ylabel('Count')
for j, v in enumerate(test_c):
    axes[1,1].text(j, v+2, f'{v}\n{v/len(y_test)*100:.0f}%', ha='center', fontsize=9)

# 6. Source breakdown
axes[1,2].axis('off')
summary_data = [
    ['NHANES', f'{len(y_nhanes):,}',    'Regression (Löwe 2008)', 'Demographics'],
    ['Colombia', f'{len(y_colombia):,}', 'Real GAD-7 ✓',           'GAD-7+PSS-10+PHQ-9'],
    ['Train (raw)', f'{len(y_train):,}', 'Mixed',                   'Stratified by source'],
    ['Train (SMOTE)', f'{len(y_train_bal):,}', 'Mixed balanced',    'SMOTE on train only'],
    ['Test', f'{len(y_test):,}',         'Mixed',                   'Natural distribution'],
]
tbl = axes[1,2].table(
    cellText=summary_data,
    colLabels=['Dataset', 'Rows', 'Labels', 'Features'],
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.8)
axes[1,2].set_title('Pipeline Summary', fontsize=11, pad=20)

for ax in axes.flat[:5]:
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figure1_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figure1_pipeline.png ✓')

In [ ]:
# ================================================================
# CELL 9 — Save All Outputs
# ================================================================
# Files produced:
#   nhanes_c3_raw.csv        — NHANES features + labels (raw)
#   colombia_c3_real.csv     — Colombia features + real labels
#   combined_c3_raw.csv      — Full combined dataset (pre-SMOTE)
#   combined_c3_balanced.csv — SMOTE-balanced training set
#   combined_c3_test.csv     — Held-out test set (natural dist)
#   feature_schema.json      — Feature column names + metadata

# NHANES raw
df_nhanes_out = X_nhanes.copy()
df_nhanes_out[TARGET_COL] = y_nhanes
df_nhanes_out['source']   = 'nhanes'
df_nhanes_out.to_csv('nhanes_c3_raw.csv', index=False)

# Colombia real
df_colombia_out = X_colombia.copy()
df_colombia_out[TARGET_COL] = y_colombia
df_colombia_out['source']   = 'colombia'
df_colombia_out.to_csv('colombia_c3_real.csv', index=False)

# Combined raw (all records, pre-SMOTE, pre-split)
df_combined_out = pd.DataFrame(X_arr, columns=FEATURE_COLS)
df_combined_out[TARGET_COL] = y_all
df_combined_out['source']   = source_all
df_combined_out.to_csv('combined_c3_raw.csv', index=False)

# Training set (SMOTE balanced)
df_train_bal = pd.DataFrame(X_train_bal, columns=FEATURE_COLS)
df_train_bal[TARGET_COL] = y_train_bal
df_train_bal.to_csv('combined_c3_balanced.csv', index=False)

# Test set (natural distribution)
df_test = pd.DataFrame(X_test, columns=FEATURE_COLS)
df_test[TARGET_COL] = y_test
df_test['source']   = src_test
df_test.to_csv('combined_c3_test.csv', index=False)

# Feature schema (needed by Phase 2)
schema = {
    'feature_cols':             FEATURE_COLS,
    'target_col':               TARGET_COL,
    'risk_labels':              {0:'Low', 1:'Medium', 2:'High'},
    'cat_feature_indices':      [1, 2],      # gender_enc, marital_enc
    'f9_index':                 8,
    'n_train_raw':              int(len(y_train)),
    'n_train_balanced':         int(len(y_train_bal)),
    'n_test':                   int(len(y_test)),
    'n_nhanes':                 int(len(y_nhanes)),
    'n_colombia':               int(len(y_colombia)),
    'colombia_source':          'Restrepo-Henao et al. 2022, Mendeley Data bytb22nf7m',
    'nhanes_gad7_method':       'Ordinal logistic regression, Löwe et al. 2008 coefficients',
    'f7_colombia_source':       'PSS-10 real stress score / 40',
    'f8_colombia_source':       'PHQ-9 real depression score / 27',
    'smote_applied':            'Training split only (never test set)'
}
with open('feature_schema.json', 'w') as f:
    json.dump(schema, f, indent=2)

print('=' * 60)
print('PHASE 1 COMPLETE ✅')
print('=' * 60)
files_out = [
    ('nhanes_c3_raw.csv',        f'{len(df_nhanes_out):,} rows',   'NHANES features + labels'),
    ('colombia_c3_real.csv',     f'{len(df_colombia_out):,} rows', 'Colombia real GAD-7'),
    ('combined_c3_raw.csv',      f'{len(df_combined_out):,} rows', 'Full combined pre-SMOTE'),
    ('combined_c3_balanced.csv', f'{len(df_train_bal):,} rows',    'SMOTE training set'),
    ('combined_c3_test.csv',     f'{len(df_test):,} rows',         'Held-out test set'),
    ('feature_schema.json',      '—',                               'Schema for Phase 2'),
    ('figure1_pipeline.png',     '—',                               'Dissertation Figure 1'),
]
print()
for fname, size, desc in files_out:
    exists = os.path.exists(fname)
    print(f'  {"✓" if exists else "✗":3s} {fname:35s} {size:20s} {desc}')

print()
print('DISSERTATION STATEMENT:')
print('  "The training dataset combines NHANES 2017–2020 demographic records')
print('   (n≈2,705, GAD-7 labels generated via ordinal logistic regression')
print('   using published coefficients, Löwe et al. 2008) with real clinical')
print('   data from 2,657 undergraduate students (Colombia, Restrepo-Henao')
print('   et al. 2022). The Colombia dataset contributes real GAD-7 labels,')
print('   validated PSS-10 stress scores (F7), and validated PHQ-9 depression')
print('   scores (F8). Both sources are stratified across train and test splits."')

In [ ]:
# ================================================================
# CELL 10 — Download All Files
# ================================================================
from google.colab import files

download_files = [
    'nhanes_c3_raw.csv',
    'colombia_c3_real.csv',
    'combined_c3_raw.csv',
    'combined_c3_balanced.csv',
    'combined_c3_test.csv',
    'feature_schema.json',
    'figure1_pipeline.png',
]

print('Downloading all Phase 1 outputs...')
for fname in download_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ✓ {fname}')
    else:
        print(f'  ✗ {fname} — not found')

print('\nUpload these files to Phase 2 when prompted.')
print('Phase 2 uses: combined_c3_raw.csv + combined_c3_balanced.csv + feature_schema.json')